# 5SC28 Design Assignment — Section 4.2.2: PPO Reference Tracking

Extends the swing-up policy (4.2.1) to a **single policy** that:
1. Swings the disc up from the bottom (θ = 0°)
2. Tracks a reference θ_ref = π ± 15° around the top position

A switching controller is not valid — this is one end-to-end trained network.

**Approach**: augment the observation with [sin θ_ref, cos θ_ref] so the policy knows its target. The reference is sampled randomly at each episode reset. PPO is used (same as 4.2.1) for training stability.

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'stable-baselines3', 'matplotlib'])


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /usr/bin/python3 -m pip install --upgrade pip


0

In [2]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

sys.path.insert(0, os.path.abspath('..'))
import gym_unbalanced_disk

/home/kaydenknapik/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
/home/kaydenknapik/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/kaydenknapik/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


## Reference-Tracking Environment Wrapper

The base environment has a 3D observation: [sin θ, cos θ, ω].

We extend it to 5D: **[sin θ, cos θ, ω, sin θ_ref, cos θ_ref]**

Using sin/cos for the reference (rather than the raw angle) avoids wrap-around discontinuities and keeps the representation consistent with the state encoding.

Reward: Gaussian centered on the reference angle:
$$r = \exp\!\left(-\frac{(\angle(\theta - \theta_\mathrm{ref}))^2}{2\sigma^2}\right)$$
where σ = π/8 ≈ 22.5°. This gives r ≈ 1 when on-target and decays smoothly away.

The reference is sampled uniformly from [π - 15°, π + 15°] at each episode reset.

In [3]:
class UnbalancedDiskRef(gym.Wrapper):
    REF_RANGE = np.radians(15)

    def __init__(self, env, obs_noise=0.05, omega_noise=0.5):
        super().__init__(env)
        low  = np.concatenate([env.observation_space.low,  [-1., -1.]]).astype(np.float32)
        high = np.concatenate([env.observation_space.high, [ 1.,  1.]]).astype(np.float32)
        self.observation_space = gym.spaces.Box(low=low, high=high, dtype=np.float32)
        self.th_ref      = np.pi
        self.obs_noise   = obs_noise
        self.omega_noise = omega_noise

    def _augment(self, obs):
        ref   = np.array([np.sin(self.th_ref), np.cos(self.th_ref)], dtype=np.float32)
        noisy = np.asarray(obs, dtype=np.float32).copy()
        noisy += np.array([
            np.random.randn() * self.obs_noise,
            np.random.randn() * self.obs_noise,
            np.random.randn() * self.omega_noise,
        ], dtype=np.float32)
        return np.concatenate([noisy, ref])

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.th_ref = np.pi + np.random.uniform(-self.REF_RANGE, self.REF_RANGE)
        return self._augment(obs), info

    def step(self, action):
        obs, _, terminated, truncated, info = self.env.step(action)
        th = np.arctan2(obs[0], obs[1])
        reward = float(0.5 * (1.0 + np.cos(th - self.th_ref)))
        return self._augment(obs), reward, terminated, truncated, info


def make_env():
    env = gym.make('unbalanced-disk-sincos-v0')
    return UnbalancedDiskRef(env)


env = make_env()
print('obs space  :', env.observation_space)
print('action space:', env.action_space)
obs, _ = env.reset()
print('sample obs :', obs)

obs space  : Box([ -1.  -1. -40.  -1.  -1.], [ 1.  1. 40.  1.  1.], (5,), float32)
action space: Box(-3.0, 3.0, (), float32)
sample obs : [-0.07357345  1.0166559   0.11903756 -0.04439571 -0.999014  ]


/home/kaydenknapik/.local/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
/home/kaydenknapik/.local/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


## PPO Training

Same PPO setup as the swing-up (4.2.1) — 4 parallel environments, 300k timesteps.
The only difference is the 5D observation and the reference-tracking reward.

If a saved model exists it is loaded directly to avoid retraining.

In [4]:
MODEL_PATH = 'ppo_reftrack.zip'

vec_env = make_vec_env(make_env, n_envs=4)
model = PPO(
    'MlpPolicy', vec_env,
    learning_rate=3e-4,
    gamma=0.99,
    ent_coef=0.005,
    n_steps=1024,
    batch_size=64,
    verbose=1,
    seed=42,
)
model.learn(total_timesteps=300_000)
model.save(MODEL_PATH)
print('Training done, model saved.')

Using cpu device


/home/kaydenknapik/.local/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
/home/kaydenknapik/.local/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/home/kaydenknapik/.local/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `step()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
/home/kaydenknapik/.local/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 300      |
|    ep_rew_mean     | 2.8      |
| time/              |          |
|    fps             | 2099     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 4096     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 300          |
|    ep_rew_mean          | 3.34         |
| time/                   |              |
|    fps                  | 1460         |
|    iterations           | 2            |
|    time_elapsed         | 5            |
|    total_timesteps      | 8192         |
| train/                  |              |
|    approx_kl            | 0.0072286846 |
|    clip_fraction        | 0.0864       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.43        |
|    explained_variance   | -8.76        |
|    learning_r

## Evaluation Across Reference Angles

Test the trained policy at several fixed reference angles: -15°, 0°, and +15° around the top.

In [5]:
def run_episode(ref_deg, n_episodes=5):
    th_ref = np.pi + np.radians(ref_deg)
    rewards = []
    for _ in range(n_episodes):
        env_e = make_env()
        obs, _ = env_e.reset()
        env_e.th_ref = th_ref
        obs[-2] = np.sin(th_ref)
        obs[-1] = np.cos(th_ref)
        total = 0.0
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, terminated, truncated, _ = env_e.step(action)
            total += r
            done = terminated or truncated
        rewards.append(total)
    return np.mean(rewards)

RUN_EVAL = False  # set to True to run simulation evaluation

if RUN_EVAL:
    for deg in [-15, -10, -5, 0, 5, 10, 15]:
        score = run_episode(deg)
        print(f'  ref = {deg:+3d}°  →  mean reward = {score:.1f} / 300')

## Episode Plots at Three Reference Angles

In [6]:
def plot_episode(ref_deg):
    th_ref = np.pi + np.radians(ref_deg)
    env_e = make_env()
    obs, _ = env_e.reset()
    env_e.th_ref = th_ref
    obs[-2] = np.sin(th_ref)
    obs[-1] = np.cos(th_ref)

    obs_list, act_list, rew_list = [], [], []
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs_list.append(obs.copy())
        act_list.append(float(action))
        obs, r, terminated, truncated, _ = env_e.step(action)
        rew_list.append(r)
        done = terminated or truncated

    obs_arr = np.array(obs_list)
    th_arr  = np.degrees(np.arctan2(obs_arr[:, 0], obs_arr[:, 1]))
    t       = np.arange(len(th_arr)) * 0.025

    fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
    axes[0].plot(t, th_arr, label='θ')
    axes[0].axhline(180 + ref_deg, color='r', ls='--', label=f'ref ({180+ref_deg}°)')
    axes[0].set_ylabel('θ [deg]'); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(t, obs_arr[:, 2], 'g')
    axes[1].set_ylabel('ω [rad/s]'); axes[1].grid(True)
    axes[2].plot(t, act_list, 'r')
    axes[2].set_ylabel('u [V]'); axes[2].set_xlabel('time [s]'); axes[2].grid(True)
    plt.suptitle(f'PPO reference tracking — ref = {180+ref_deg}°')
    plt.tight_layout()
    plt.show()
    print(f'Total reward: {sum(rew_list):.1f} / {len(rew_list)}')

if RUN_EVAL:
    for deg in [-15, 0, 15]:
        plot_episode(deg)

## Live Demo on Real Hardware\n\nMake sure the disc is at the **bottom position** before running. Connects over USB, swings up to 180°, holds for 1.5s, then tracks a ±15° sine wave reference. Interrupt the kernel to stop.

In [7]:
import time
import pygame

env_live = gym_unbalanced_disk.UnbalancedDisk_exp_sincos()
obs, _ = env_live.reset()

def make_obs(raw_obs, th_ref):
    ref = np.array([np.sin(th_ref), np.cos(th_ref)], dtype=np.float32)
    return np.concatenate([np.asarray(raw_obs, dtype=np.float32), ref])

def draw_overlay(env, phase, th_ref, th):
    if not hasattr(env, '_font'):
        pygame.font.init()
        env._font = pygame.font.SysFont('monospace', 18, bold=True)
    lines = [
        f'Phase:  {phase}',
        f'Target: {np.degrees(th_ref):+.1f} deg',
        f'Angle:  {np.degrees(th):+.1f} deg',
    ]
    for i, line in enumerate(lines):
        surf = env._font.render(line, True, (30, 30, 30))
        env.viewer.blit(surf, (10, 10 + i * 22))
    pygame.display.flip()

AMP           = np.radians(15)
PERIOD        = 6.0
HOLD_TOP_SECS = 1.5

phase       = 'swing-up'
phase_start = time.time()

print('Phase: swing-up (target = 180°)')
try:
    while True:
        t_now    = time.time()
        th       = np.arctan2(obs[0], obs[1])
        near_top = abs((th - np.pi + np.pi) % (2*np.pi) - np.pi) < np.radians(25)

        if phase == 'swing-up' and near_top:
            phase, phase_start = 'holding', t_now
            print('Phase: holding at 180°...')
        elif phase == 'holding' and (t_now - phase_start) >= HOLD_TOP_SECS:
            phase, phase_start = 'tracking', t_now
            print('Phase: tracking sine wave ±15°')

        th_ref = np.pi + AMP * np.sin(2 * np.pi * (t_now - phase_start) / PERIOD) if phase == 'tracking' else np.pi

        action, _ = model.predict(make_obs(obs, th_ref), deterministic=True)
        obs, _, terminated, truncated, _ = env_live.step(action)
        env_live.render()
        draw_overlay(env_live, phase, th_ref, th)

        if terminated or truncated:
            print('Episode ended, restarting...')
            obs, _ = env_live.reset()
            phase, phase_start = 'swing-up', time.time()
            print('Phase: swing-up (target = 180°)')
finally:
    env_live.close()

AttributeError: 'NoneType' object has no attribute 'set_configuration'